In [ ]:
# Load in merged data frame 
# Runs merge_station_events 

import pandas as pd
import numpy as np
import os
import math

from src.cleaning.clean_stations import clean_station_data
from src.cleaning.merge_station_events import merge_station_events
from src.cleaning.merged_tornado_indicator import create_tornado_indicator

columns_to_drop = ['NAME',
                    'SOURCE',
                    'REPORT_TYPE',
                    'CALL_SIGN',
                    'QUALITY_CONTROL',
                    'CALL_SIGN.1',
                    'QUALITY_CONTROL.1',
                    'REPORT_TYPE.1',
                    'SOURCE.1',
                    'AB1',
                    'AD1',
                    'AE1',
                    'AG1',
                    'AH1',
                    'AH2',
                    'AH3',
                    'AH4',
                    'AH5',
                    'AH6',
                    'AI1',
                    'AI2',
                    'AI3',
                    'AI4',
                    'AI5',
                    'AI6',
                    'AK1',
                    'AM1',
                    'AN1',
                    'AT1',
                    'AT2',
                    'AT3',
                    'AT4',
                    'AT5',
                    'AT6',
                    'AT7',
                    'AT8',
                    'AU1',
                    'AU2',
                    'AU3',
                    'AU4',
                    'AU5',
                    'AW1',
                    'AW2',
                    'AW3',
                    'AW4',
                    'AW5',
                    'AW6',
                    'AW7',
                    'AX1',
                    'AX2',
                    'AX3',
                    'AX4',
                    'AX5',
                    'AX6',
                    'ED1',
                    'EQD',
                    'GD1',
                    'GD2',
                    'GD3',
                    'GD4',
                    'GE1',
                    'GF1',
                    'IA1',
                    'KC1',
                    'KC2',
                    'KD1',
                    'KD2',
                    'KE1',
                    'MH1',
                    'MV1',
                    'MW1',
                    'MW2',
                    'MW3',
                    'MW4',
                    'MW5',
                    'OE1',
                    'OE2',
                    'OE3',
                    'REM',
                    'SA1',
                    'UA1',
                    'UG1',
                    'WA1'
                    ]
### Reasons to get rid of 
# 'NAME': already have an identifier column 'STATION'.
# 'SOURCE': this is just the source or sources used to create sample.
# 'REPORT_TYPE': denotes the type of geophysical surface observation.
# 'CALL_SIGN': call letters assigned to a weather station. We already have an identifier.
# 'QUALITY_CONTROl': For predicting tornadoes, this might not be super useful.
# though it may be good to keep in mind if so desired. One can erase all V01 
# entries (no quality control).
# 'CALL_SIGN.1': see CALL_SIGN.
# 'QUALITY_CONTROL.1' : see QUALITY_CONTROL.
# 'REPORT_TYPE.1': see REPORT_TYPE.
# 'SOURCE.1': see SOURCE.
# 'AB1': Liquid Precipitation Monthly total-- too long of a time scale.
# 'AD1: Liquid Precipitation Greatest Amount in 24 Hours, For the month -- too long of a time scale.
# 'AE1': Number of Days with Specific Amounts for Each Month -- Can be obtained through AA1-AA4
# 'AG1': 'Precipitation Estimated Observation -- not sure how this is different from AA1-AA4
# 'AH1'-- AH6' : Liquid Precipitation Maximum Short Duration, For The Month -- too long of a time scale.
# 'AI1 -- AI6' : Identical to 'AH1'--'AH6'
# 'AK1 Greatest Snow Depth on Ground for the Month
# 'AM1':
# 'AN1':
# 'AT1--AT8': Data leakage
# 'AU1--AU5': Data leakage 
# 'AW1--AW7': Data leakage
# 'AX1--AX6': Data leakage
# 'ED1': Runway Visibility
# 'EQD':
# 'GD1--GD4': Similar to GA1-GA6
# 'GE1': Similar to GA1-GA6 (may include later)
# 'GF1': Similar to GA1-GA6 (may include later)
# 'IA1': 
# 'KC1--KD2': too long of a time scale
# 'KE1': Extreme Temperatures, Number of Days Exceeding Criteria, For the Month -- too long of a time scale
# 'MH1': Atmospheric Pressure Observation for the month -- too long of a time scale.
# 'MK1' : See 'MH1'
# 'MV1 : Present Weather in Vicinity Observation -- Potential Data Leakage
# 'MW1--MW5' : Present Weather Observation -- Potential Data Leakage 
# 'OE1--OE3': Already have WND
# 'REM' : These are remarks
# 'SA1': Sea Surface Temperature 
# 'UA1'--'UG1' Marine Data?
# 'WA1'-- Platform Ice accretion ###


# SPLIT TUPLES IN PARTICULAR COLUMNS

# These are ordered in the way their tuples are ordered

mapping = {
'AA1':['AA1- Liquid Precipitation- Period Quantity in Hours', 
       'AA1- Liquid Precipitation- Depth Dimension',
       'AA1- Liquid Precipitation- Condition Code',
       'AA1- Quality Code'],

'AA2': ['AA2- Liquid Precipitation- Period Quantity in Hours', 
       'AA2- Liquid Precipitation- Depth Dimension',
       'AA2- Liquid Precipitation- Condition Code',
       'AA2- Quality Code'],

'AA3' : ['AA3- Liquid Precipitation- Period Quantity in Hours', 
       'AA3- Liquid Precipitation- Depth Dimension',
       'AA3- Liquid Precipitation- Condition Code',
       'AA3- Quality Code'],

'AA4': ['AA4- Liquid Precipitation- Period Quantity in Hours', 
       'AA4- Liquid Precipitation- Depth Dimension',
       'AA4- Liquid Precipitation- Condition Code',
       'AA4- Quality Code'],

'AJ1' : ['AJ1- Snow Depth- Dimension',
       'AJ1- Snow Depth- Condition Code',
       'AJ1- Snow Depth- Quality Code',
       'AJ1- Snow Depth- Equivalent Water Depth Dimension',
       'AJ1- Snow Depth- Equivalent Water Condition Code',
       'AJ1- Snow Depth- Equivalent Water Condition Quality Code'],

'AL1' : ['AL1- Snow Accumulation- Period Quantity',
       'AL1- Snow Accumulation- Depth Dimension',
       'AL1- Snow Accumulation- Condition Code',
       'AL1- Snow Accumulation- Quality Code'],

'CIG': ['CIG- Sky Condition Observation- Ceiling Height Dimension',
       'CIG- Sky Condition Observation- Ceiling Quality Code',
       'CIG- Sky Condition Observation- Ceiling Determination Code',
       'CIG- Sky Condition Observation- Cavok Code'],

'DEW':['DEW- Air Temperature Observation- Dew Point Temperature',
       'DEW- Air Temperature Observation- Dew Point Quality Code'],

'GA1':['GA1- Sky Cover Layer- Coverage Code',
       'GA1- Sky Cover Layer- Coverage Quality Code',
       'GA1- Sky Cover Layer- Base Height Dimensions',
       'GA1- Sky Cover Layer- Base Height Quality Code',
       'GA1- Sky Cover Layer- Cloud Type Code',
       'GA1- Sky Cover Layer- Cloud Type Quality Code'],

'GA2':['GA2- Sky Cover Layer- Coverage Code',
       'GA2- Sky Cover Layer- Coverage Quality Code',
       'GA2- Sky Cover Layer- Base Height Dimensions',
       'GA2- Sky Cover Layer- Base Height Quality Code',
       'GA2- Sky Cover Layer- Cloud Type Code',
       'GA2- Sky Cover Layer- Cloud Type Quality Code'],

'GA3':['GA3- Sky Cover Layer- Coverage Code',
       'GA3- Sky Cover Layer- Coverage Quality Code',
       'GA3- Sky Cover Layer- Base Height Dimensions',
       'GA3- Sky Cover Layer- Base Height Quality Code',
       'GA3- Sky Cover Layer- Cloud Type Code',
       'GA3- Sky Cover Layer- Cloud Type Quality Code'],

'GA4':['GA4- Sky Cover Layer- Coverage Code',
       'GA4- Sky Cover Layer- Coverage Quality Code',
       'GA4- Sky Cover Layer- Base Height Dimensions',
       'GA4- Sky Cover Layer- Base Height Quality Code',
       'GA4- Sky Cover Layer- Cloud Type Code',
       'GA4- Sky Cover Layer- Cloud Type Quality Code'],

'GA5':['GA5- Sky Cover Layer- Coverage Code',
       'GA5- Sky Cover Layer- Coverage Quality Code',
       'GA5- Sky Cover Layer- Base Height Dimensions',
       'GA5- Sky Cover Layer- Base Height Quality Code',
       'GA5- Sky Cover Layer- Cloud Type Code',
       'GA5- Sky Cover Layer- Cloud Type Quality Code'],

'GA6':['GA6- Sky Cover Layer- Coverage Code',
       'GA6- Sky Cover Layer- Coverage Quality Code',
       'GA6- Sky Cover Layer- Base Height Dimensions',
       'GA6- Sky Cover Layer- Base Height Quality Code',
       'GA6- Sky Cover Layer- Cloud Type Code',
       'GA6- Sky Cover Layer- Cloud Type Quality Code'],


'GJ1':['GJ1- Sunshine Observation- Sunshine Duration Quantity',
       'GJ1- Sunshine Observation- Sunshine Duration Quality Code'],

'GK1': ['GK1- Sunshine Observation- Percent of Possible Sunshine Quantity',
       'GK1- Sunshine Observation- Percent of Possible Sunshine Quality Code'],

'GP1' : ['GP1- Modeled Solar Irradiance Section- Time Period in Minutes',
       'GP1- Modeled Solar Irradiance Section- Modeled Global Horizontal',
       'GP1- Modeled Solar Irradiance Section- Modeled Global Horizontal Source Flag',
       'GP1- Modeled Solar Irradiance Section- Modeled Global Horizontal Uncertainty',
       'GP1- Modeled Solar Irradiance Section- Modeled Direct Normal',
       'GP1- Modeled Solar Irradiance Section- Modeled Direct Normal Source Flag',
       'GP1- Modeled Solar Irradiance Section- Modeled Direct Normal Uncertainty',
       'GP1- Modeled Solar Irradiance Section- Modeled Diffuse Horizontal',
       'GP1- Modeled Solar Irradiance Section- Modeled Diffuse Horizontal Source Flag',
       'GP1- Modeled Solar Irradiance Section- Time Period in Minutes'],

'GQ1':['GQ1- Hourly Solar Angle Section- Hourly Solar Angle Time Period',
       'GQ1- Hourly Solar Angle Section- Hourly Mean Zenith Angle',
       'GQ1- Hourly Solar Angle Section- Hourly Mean Zenith Angle Quality Code',
       'GQ1- Hourly Solar Angle Section- Hourly Mean Azimuth Angle',
       'GQ1- Hourly Solar Angle Section- Hourly Mean Azimuth Angle Quality Code'],

'GR1':['GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation Time Period',
       'GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation on a Horizontal Surface',
       'GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation on a Horizontal Surface Quality Code',
       'GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation Normal to the Sun',
       'GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation Normal to the Sun Quality Code'],

'HL1':['HL1- Hail- Size',
       'HL1- Hail- Size Quality Code'],

'KA1': ['KA1- Extreme Air Temperature- Period Quantity',
       'KA1- Extreme Air Temperature- Code',
       'KA1- Extreme Air Temperature- Air Temperature',
       'KA1- Extreme Air Temperature- Temperature Quality Code'],
'KA2': ['KA2- Extreme Air Temperature- Period Quantity',
       'KA2- Extreme Air Temperature- Code',
       'KA2- Extreme Air Temperature- Air Temperature',
       'KA2- Extreme Air Temperature- Temperature Quality Code'],
'KA3': ['KA3- Extreme Air Temperature- Period Quantity',
       'KA3- Extreme Air Temperature- Code',
       'KA3- Extreme Air Temperature- Air Temperature',
       'KA3- Extreme Air Temperature- Temperature Quality Code'],
'KA4': ['KA4- Extreme Air Temperature- Period Quantity',
       'KA4- Extreme Air Temperature- Code',
       'KA4- Extreme Air Temperature- Air Temperature',
       'KA4- Extreme Air Temperature- Temperature Quality Code'],
'KB1': ['KB1- Average Air Temperature- Period Quantity',
       'KB1- Average Air Temperature- Type Code',
       'KB1- Average Air Temperature- Air Temperature',
       'KB1- Average Air Temperature- Temperature Quality Code'],
'KB2': ['KB2- Average Air Temperature- Period Quantity',
       'KB2- Average Air Temperature- Type Code',
       'KB2- Average Air Temperature- Air Temperature',
       'KB2- Average Air Temperature- Temperature Quality Code'],
'KB3': ['KB3- Average Air Temperature- Period Quantity',
       'KB3- Average Air Temperature- Type Code',
       'KB3- Average Air Temperature- Air Temperature',
       'KB3- Average Air Temperature- Temperature Quality Code'],

'KG1':['KG1- Average Dew Point and Wet Bulb Temperature- Period Quantity',
       'KG1- Average Dew Point and Wet Bulb Temperature- Code',
       'KG1- Average Dew Point and Wet Bulb Temperature- Temperature',
       'KG1- Average Dew Point and Wet Bulb Temperature- Derived Code',
       'KG1- Average Dew Point and Wet Bulb Temperature- Quality Code'],

'KG2':['KG2- Average Dew Point and Wet Bulb Temperature- Period Quantity',
       'KG2- Average Dew Point and Wet Bulb Temperature- Code',
       'KG2- Average Dew Point and Wet Bulb Temperature- Temperature',
       'KG2- Average Dew Point and Wet Bulb Temperature- Derived Code',
       'KG2- Average Dew Point and Wet Bulb Temperature- Quality Code'],

'MA1':['MA1-Atmospheric Pressure Observation- Altimeter Setting Rate',
       'MA1-Atmospheric Pressure Observation- Altimeter Quality Code',
       'MA1-Atmospheric Pressure Observation- Station Pressure Rate',
       'MA1-Atmospheric Pressure Observation- Station Pressure Quality Code'],

'MD1':['MD1- Atmospheric Pressure Change- Tendency Code',
       'MD1- Atmospheric Pressure Change- Quality Tendency Code',
       'MD1- Atmospheric Pressure Change- Three Hour Quantity',
       'MD1- Atmospheric Pressure Change- Quality Three Hour Code',
       'MD1- Atmospheric Pressure Change- Twenty Four Hour Quantity',
       'MD1- Atmospheric Pressure Change- Quality Twenty Four Hour Code'],

'MF1': ['MF1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure for the Day (Derived)',
       'MF1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure Quality Code',
       'MF1- Atmospheric Pressure Observation (SLP/STP)- Average Sea Level Pressure for the Day',
       'MF1- Atmospheric Pressure Observation (SLP/STP)- Average Sea Level Pressure Quality Code'],

'MG1': ['MG1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure for the Day (Derived)',
       'MG1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure Quality Code',
       'MG1- Atmospheric Pressure Observation (SLP/STP)- Minimum Sea Level Pressure for the Day',
       'MG1- Atmospheric Pressure Observation (SLP/STP)- Minimum Sea Level Pressure Quality Code'],

'OC1':['OC1- Wind Gust Observation- Speed Rate',
       'OC1- Wind Gust Observation- Quality Code'],

'RH1':['RH1- Relative Humidity- Period Quantity',
       'RH1- Relative Humidity- Code',
       'RH1- Relative Humidity- Percentage',
       'RH1- Relative Humidity- Derived Code',
       'RH1- Relative Humidity- Quality Code'],

'RH2':['RH2- Relative Humidity- Period Quantity',
       'RH2- Relative Humidity- Code',
       'RH2- Relative Humidity- Percentage',
       'RH2- Relative Humidity- Derived Code',
       'RH2- Relative Humidity- Quality Code'],

'RH3':['RH3- Relative Humidity- Period Quantity',
       'RH3- Relative Humidity- Code',
       'RH3- Relative Humidity- Percentage',
       'RH3- Relative Humidity- Derived Code',
       'RH3- Relative Humidity- Quality Code'],

'SLP':['SLP- Atmospheric Pressure Observation- Sea Level Pressure',
       'SLP- Atmospheric Pressure Observation- Sea Level Pressure Quality Code'],

'TMP':['TMP- Air Temperature Observation- Air Temperature',
       'TMP- Air Temperature Observation- Air Temperature Quality Code'],

'VIS': ['VIS- Visibility Observation- Distance Dimension',
       'VIS- Visibility Observation- Distance Quality Code',
       'VIS- Visibility Observation- Variability Code',
       'VIS- Visibility Observation- Quality Variability Code'],

'WND':['WND- Wind Observation- Direction Angle',
       'WND- Wind Observation- Direction Quality Code',
       'WND- Wind Observation- Type Code',
       'WND- Wind Observation- Speed Rate',
       'WND- Wind Observation- Speed Quality Code'],
}

# The following creates merged dataframe where we count tornadoes as occurring if
# 1.) The station makes an observation within time_window hours of the tornadoes beginning time,
# 2.) and the station is within val_radius KILOMETERS of the tornadoes touchdown point.
# It will take a while for the cell to finish running.
# See src/cleaning for py files that are utilized in create_tornado_indicator
# They are not well documented as of now-- sorry!

data =create_tornado_indicator(drop_cols=columns_to_drop,split_tuples=True, mapping=mapping, drop_originals=True, tuple_sep=',',time_window=1,val_radius=50)

/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_14696/1313703029.py:32: DtypeWarning: Columns (7,14,15,16,17,19,20,21,22,23,24,25,26,27,28,29,30,32,33,34,40,41,42,43,44,45,46,47,50,51,52,56,57,58,59,60,65,68,69,70,71,76,79,80,81,82,83,90,91,92,93,94,95,96,97,98,99,102,104,105,106,110,111,113,120,121,122,125) have mixed types. Specify dtype option on import or set low_memory=False.
  station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_14696/1313703029.py:32: DtypeWarning: Columns (39,40,41,42,43,47,48,52,53,54,55,57,58,59,60,61,65,70,71,76,77,88,89,106,108,109,110) have mixed types. Specify dtype option on import or set low_memory=False.
  station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_14696/1313703029.py:32: DtypeWarning: Columns (15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,32,34,40,41,4